# 06. 레이아웃 분석 + VLM 파싱 — PP-DocLayoutV2 · PaddleOCR-VL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/07_layout_vlm_parsing.ipynb)

> 🧭 **스토리** — 06에서 OCR은 글자는 줬지만 "여기가 제목, 여기가 표"는 몰랐다. 이번에는 **레이아웃 모델**로 문서의 뼈대와 읽기 순서를 먼저 잡고, 이어서 **VLM(비전-언어 모델)** 에게 레이아웃 모델이 나눈 영역을 읽혀 마크다운으로 바꾸게 한다.
>
> 📖 **이론 연결** — 이론2 p16~25 (레이아웃 분석·표 구조 인식·Docling 구조 파싱·Document VLM·VLM 선택 기준) · ⏱ **16분** · 🖥 **GPU 필수**
>
> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. (GPU가 없으면 작은 설정으로 느리게 돌아갑니다.)
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
PaddlePaddle(딥러닝 프레임워크)과 PaddleOCR(레이아웃·VLM 파이프라인), 비교용 EasyOCR을 설치합니다. **GPU가 잡혀 있으면 CUDA 12.6용 휠**, 없으면 CPU 휠을 고릅니다. 설치 시간이 출력됩니다(코랩 T4 기준 보통 1~3분).
> 설치가 끝나고 "restart" 안내가 떠도 **재시작하지 말고** 다음 셀로 넘어가세요. 설치나 불러오기가 실패해도 노트북은 **미리 뽑아 둔 결과(폴백)** 로 끝까지 진행됩니다.

In [ ]:
import time, torch
_t0 = time.time()
RUN_GRANITE = False     # 🟠 9절 Granite-Docling을 돌리려면 True로 바꾸고 이 셀부터 다시 실행(추가 설치 1~2분)
if torch.cuda.is_available():
    %pip -q install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
else:
    %pip -q install paddlepaddle==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip -q install -U "paddleocr[doc-parser]" easyocr
if RUN_GRANITE:
    %pip -q install docling
print(f"설치 소요 {time.time() - _t0:.0f}초")

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ Paddle 불러오기 점검 + 채점 도구(05와 같은 셀 일치율)
import os, re, glob, time, pandas as pd
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")   # 모델 저장소 연결 점검 생략 → 시작이 빨라짐
try:
    import paddle
    from paddleocr import LayoutDetection, PaddleOCRVL
    PADDLE_OK = True
    print(f"paddle {paddle.__version__} · CUDA 빌드: {paddle.device.is_compiled_with_cuda()}")
except Exception as e:
    PADDLE_OK = False
    print("⚠️ Paddle 불러오기 실패 → 폴백으로 진행합니다:", type(e).__name__, str(e)[:200])

def norm(x):
    return re.sub(r"\s+|\*\*", "", "" if x is None else str(x))

def match_rate(name, rows):
    """정답 CSV 대비 셀 일치율(0~1) — 05 노트북과 같은 방식"""
    df = pd.read_csv(fetch(f"tables/{name}.csv"), dtype=str).fillna("")
    gt = [[norm(c) for c in r] for r in [list(df.columns)] + df.values.tolist()]
    rows = [[norm(c) for c in r] for r in (rows or [])]
    if not rows:
        return 0.0
    hit = sum(max(sum(1 for j, c in enumerate(g) if j < len(r) and r[j] == c) for r in rows) for g in gt)
    return hit / sum(len(g) for g in gt)

SCAN = fetch("scan/p2.png")       # 06에서 OCR로 읽었던 그 스캔본(2쪽, 국내 출장)

## 1. 레이아웃 분석 — PP-DocLayoutV2 🔵
PP-DocLayoutV2(204MB, Apache 2.0)는 페이지에서 **25종류의 영역**(문서 제목·문단 제목·본문·표·그림·머리글·바닥글·쪽 번호 …)을 찾고, 포인터 네트워크로 **읽기 순서**까지 매깁니다(이론2 p16). 결과는 `boxes` 목록이며 각 박스는 `cls_id, label, score, coordinate [x1, y1, x2, y2]`를 가집니다.

In [ ]:
# 강사 PC(CPU)에서 미리 뽑아 둔 2쪽 결과 — Paddle을 못 쓸 때만 사용하는 폴백
LAYOUT_FALLBACK = [
    {"cls_id": 12, "label": "header",          "score": 0.93, "coordinate": [173, 44, 610, 79],    "order": None},
    {"cls_id": 17, "label": "paragraph_title", "score": 0.95, "coordinate": [184, 180, 464, 230],  "order": 1},
    {"cls_id": 22, "label": "text",            "score": 0.97, "coordinate": [181, 258, 1468, 355], "order": 2},
    {"cls_id": 22, "label": "text",            "score": 0.98, "coordinate": [179, 367, 1457, 502], "order": 3},
    {"cls_id": 22, "label": "text",            "score": 0.96, "coordinate": [176, 519, 1399, 607], "order": 4},
    {"cls_id": 7,  "label": "figure_title",    "score": 0.93, "coordinate": [176, 625, 613, 666],  "order": None},
    {"cls_id": 21, "label": "table",           "score": 0.99, "coordinate": [161, 675, 1502, 1079], "order": None},
    {"cls_id": 16, "label": "number",          "score": 0.81, "coordinate": [792, 2236, 821, 2265], "order": 5},
]
boxes = LAYOUT_FALLBACK
if PADDLE_OK:
    try:
        t0 = time.time()
        layout_model = LayoutDetection(model_name="PP-DocLayoutV2")      # 첫 실행 때 모델 다운로드(204MB)
        out = layout_model.predict(SCAN, batch_size=1, layout_nms=True)
        for r in out:
            r.print()                                  # 결과 딕셔너리 출력
            r.save_to_json("out_layout")               # out_layout/p2_res.json 저장
        boxes = json.load(open(glob.glob("out_layout/*.json")[0], encoding="utf-8"))["boxes"]
        print(f"\n레이아웃 분석 {time.time() - t0:.1f}초 (모델 로드 포함)")
    except Exception as e:
        print("⚠️ 레이아웃 모델 실패 → 폴백 결과 사용:", type(e).__name__, str(e)[:200])
else:
    print("(폴백) 미리 뽑아 둔 레이아웃 결과를 씁니다.")

layout_df = pd.DataFrame([{"순서": i, "label": b["label"], "score": round(float(b["score"]), 2),
                           "order": b.get("order"), "coordinate": [int(v) for v in b["coordinate"]]}
                          for i, b in enumerate(boxes)])
display(layout_df)

💡 **해석** — 강사 PC(CPU)에서 2쪽은 **박스 8개**(머리글 1 · 문단 제목 1 · 본문 3 · 표 캡션 1 · 표 1 · 쪽 번호 1)였고, 첫 실행(모델 다운로드 포함) 약 14초, 이후 약 4초였습니다. 몇 가지가 눈에 띕니다.
- 표 캡션 "표 1. …"은 `figure_title`(그림·표 제목)로 분류됩니다.
- `order`는 **본문 흐름에 속한 영역에만** 번호가 붙습니다. 머리글·표·캡션은 `None`이고, 목록 자체가 위→아래 읽기 순서로 정렬되어 있습니다.
- 06에서 pdfplumber가 못 찾은 **무괘선 표(3쪽)도 이 모델은 `table` 박스로 잡습니다**. 선이 아니라 "생김새"를 보기 때문입니다.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
PALETTE = {"paragraph_title": (200, 0, 0), "doc_title": (200, 0, 0), "text": (0, 90, 200), "table": (0, 150, 0),
           "figure_title": (220, 120, 0), "header": (130, 130, 130), "footer": (130, 130, 130), "number": (130, 130, 130)}
try:   # 한글 라벨용 폰트(데이터 헬퍼가 받아 둔 BMDOHYEON) — 없으면 기본 폰트
    FONT = ImageFont.truetype(str(CACHE / "BMDOHYEON.ttf"), 34)
except Exception:
    FONT = ImageFont.load_default()

page = Image.open(SCAN).convert("RGB")
vis = page.copy(); d = ImageDraw.Draw(vis)
for i, b in enumerate(boxes):
    x1, y1, x2, y2 = map(int, b["coordinate"])
    col = PALETTE.get(b["label"], (150, 0, 150))
    d.rectangle([x1, y1, x2, y2], outline=col, width=5)
    d.text((x1 + 6, y1 + 4), f"{i} {b['label']}", fill=col, font=FONT)
display(vis.crop((0, 0, vis.width, 1150)).resize((vis.width // 2, 575)))   # 위쪽 절반만 축소 표시

## 2. 레이아웃 순서대로 OCR 다시 조립하기 🔵
레이아웃 박스를 순서대로 잘라 **박스마다** EasyOCR을 돌립니다. 본문 박스는 한 문단으로 합치고, 표 박스는 06에서 만든 "y좌표로 행 묶기"를 적용합니다. 머리글·쪽 번호는 RAG에 쓸모없으니 버립니다.

In [ ]:
import easyocr
reader = easyocr.Reader(["ko", "en"], gpu=HAS_GPU)
SKIP = {"header", "footer", "number"}            # 검색에 방해만 되는 영역

def rows_by_y(items, tol=30):
    rows, start = [], None
    for b, t, c in sorted(items, key=lambda x: x[0][0][1]):
        if start is None or b[0][1] - start > tol:
            rows.append([]); start = b[0][1]
        rows[-1].append((b[0][0], t))
    return [[t for x, t in sorted(r)] for r in rows]

t0 = time.time()
assembled, layout_table_rows = [], []
for i, b in enumerate(boxes):
    if b["label"] in SKIP:
        continue
    x1, y1, x2, y2 = map(int, b["coordinate"])
    crop = np.array(page.crop((max(x1 - 8, 0), max(y1 - 8, 0), x2 + 8, y2 + 8)))
    if b["label"] == "table":
        layout_table_rows = rows_by_y(reader.readtext(crop))
        assembled += [f"[{i} table] " + " | ".join(r) for r in layout_table_rows]
    else:
        assembled.append(f"[{i} {b['label']}] " + " ".join(reader.readtext(crop, detail=0, paragraph=True)))
print(f"박스별 OCR {time.time() - t0:.1f}초\n")
print("\n".join(a[:150] for a in assembled))

In [ ]:
plain = reader.readtext(SCAN, detail=0)            # 05 방식: 페이지 전체를 한 번에 OCR
n = 15
side = pd.DataFrame({"05 방식 — 페이지 통째 OCR (줄 조각)": (plain + [""] * n)[:n],
                     "06 방식 — 레이아웃 순서 + 박스별 OCR": (assembled + [""] * n)[:n]})
pd.set_option("display.max_colwidth", 70)
display(side)
print(f"표 셀 일치율 — 레이아웃 표 박스 + 행 묶기: {match_rate('t1_domestic', layout_table_rows):.0%}")

💡 **해석** — 왼쪽은 **줄 조각**의 나열입니다. 문장이 줄 끝에서 끊기고, 머리글(`한빛l크 인사팀`, `HR-2026-03`)이 본문과 섞여 있고, 표 셀은 낱개 숫자로 흩어집니다. 오른쪽은 박스마다 **역할 이름표**가 붙고, 본문은 문단 단위로 이어지고, 표는 행 단위로 묶였습니다. 이 페이지는 단(段)이 하나라 순서 자체는 비슷하지만, **2단 논문·신문·양식**에서는 읽기 순서 모델이 없으면 왼쪽·오른쪽 단이 뒤섞입니다(이론2 p17).

단, 글자 인식은 여전히 EasyOCR이라 `구성하다`, `원 일` 같은 오류는 그대로입니다. 표 셀 일치율도 05와 비슷한 수준(강사 환경 86%, 06는 89%)입니다. **구조는 레이아웃 모델이, 글자 품질은 인식 모델이** 결정합니다.

## 3. PaddleOCR-VL — 스캔 한 장을 통째로 마크다운으로 🔵
PaddleOCR-VL(0.9B, Apache 2.0)은 레이아웃 모델(PP-DocLayoutV2)로 영역을 나눈 뒤, 각 영역을 **작은 VLM**이 읽어 마크다운·표(HTML)로 바꿉니다. **v1은 한국어를 포함한 109개 언어를 명시**하므로 `pipeline_version="v1"`을 씁니다(1.5·1.6도 시리즈 차원에서 109개 언어를 명시하지만, 버전별 한국어 정확도는 따로 재야 하므로 실습은 검증한 v1로 고정합니다. 이론2 p24).

In [ ]:
def gt_page2():
    gt = load_text("hanbit_rules_gt.md")
    return gt[gt.find("## 제2장"):gt.find("## 제3장")]

VL_MD, vl = None, None
if PADDLE_OK:
    try:
        t0 = time.time()
        vl = PaddleOCRVL(pipeline_version="v1")              # 첫 실행 때 모델 다운로드(약 2GB)
        print(f"모델 로드 {time.time() - t0:.0f}초")
        t0 = time.time()
        for res in vl.predict(SCAN):
            res.save_to_json("out_vl")
            res.save_to_markdown("out_vl")
        print(f"1쪽 변환 {time.time() - t0:.0f}초 ({DEVICE})")
        VL_MD = open(glob.glob("out_vl/*.md")[0], encoding="utf-8").read()
    except Exception as e:
        print("⚠️ PaddleOCR-VL 실패:", type(e).__name__, str(e)[:300])
VL_IS_FALLBACK = VL_MD is None
if VL_IS_FALLBACK:
    print("(폴백) VL 결과 대신 **정답 마크다운**의 2쪽 부분을 씁니다 — 아래 수치는 VL 성능이 아닙니다.")
    VL_MD = gt_page2()
print("\n" + re.sub(r"<td[^>]*>", "<td>", VL_MD)[:1600])       # 표 HTML의 style 속성은 줄여서 출력

💡 **해석** — 강사 PC(Mac CPU)에서 모델 로드 약 30초, 1쪽 변환 47초~3분(부하에 따라)이었습니다. T4에서는 보통 훨씬 빠릅니다. 결과를 06의 EasyOCR과 비교해 보세요. `구성한다`, `영수증을`, `택시를`처럼 **받침까지 정확**하고, 줄 끝에서 끊겼던 문장이 한 문단으로 이어졌고, 머리글·쪽 번호는 빠졌습니다. 제목은 `## 제2장 국내 출장`으로, 표는 `<table>` HTML로 나옵니다.

In [ ]:
def html_rows(html):
    """<table> HTML → [[셀, …], …]"""
    return [[re.sub(r"<[^>]+>", "", c).strip() for c in re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", tr, flags=re.S)]
            for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", html, flags=re.S)]

def md_rows(md):
    """마크다운 | 표 | → [[셀, …], …] (폴백 정답 마크다운용)"""
    return [[c.strip() for c in l.strip().strip("|").split("|")] for l in md.splitlines()
            if l.strip().startswith("|") and not re.fullmatch(r"[|\-: ]+", l.strip())]

vl_rows = html_rows(VL_MD) or md_rows(VL_MD)
if vl_rows:
    display(pd.DataFrame(vl_rows[1:], columns=vl_rows[0]))
summary = pd.Series({"05 EasyOCR + 행 묶기(tol=30)": 0.89,     # 05 노트북 강사 환경 값
                     "06 레이아웃 박스 + EasyOCR": match_rate("t1_domestic", layout_table_rows),
                     ("(폴백) 정답 마크다운" if VL_IS_FALLBACK else "06 PaddleOCR-VL v1"): match_rate("t1_domestic", vl_rows)}, name="표 1 셀 일치율")
display(summary.map("{:.0%}".format).to_frame())

💡 **해석** — 강사 환경에서 PaddleOCR-VL의 표 1은 **100%**(24/24셀)였습니다. 머리글의 `원/일`의 `/`까지 정확합니다. 스캔본이라도 **한국어를 지원하는 VLM**이면 디지털 PDF 파서와 같은 품질에 도달할 수 있다는 뜻입니다. 대가는 **모델 크기(0.9B)·GPU·시간**입니다. 수백 쪽을 처리하면 비용이 쌓입니다.

## 4. 바꿔서 실행 🟡
1. **`layout_nms` 끄기** — 겹치는 박스를 정리하는 후처리(NMS)를 끄면 박스가 늘어날까요? `SCAN_Y`를 표가 2개인 `scan/p4.png`나 `scan/p5.png`로 바꿔서도 해 보세요.
2. **표만 잘라서 VLM에** — 레이아웃이 찾은 표 박스만 잘라 PaddleOCR-VL에 넣습니다. 통째로 넣을 때와 결과·시간이 어떻게 다른가요?

> 강사 환경에서는 이 규정집 페이지들의 NMS 켜기/끄기 결과가 같았습니다(박스가 겹칠 일이 적은 단순한 쪽). 표만 잘라 넣으면 표 1 일치율은 그대로 100%, 시간은 통째 변환의 절반 이하였습니다.

In [ ]:
SCAN_Y = "scan/p4.png"                     # ← 바꿔 보세요
if PADDLE_OK and "layout_model" in globals():
    for nms in [True, False]:
        bx = next(iter(layout_model.predict(fetch(SCAN_Y), batch_size=1, layout_nms=nms)))["boxes"]
        print(f"layout_nms={nms!s:<5} → 박스 {len(bx)}개:", pd.Series([b["label"] for b in bx]).value_counts().to_dict())
else:
    print("Paddle 없음 — 건너뜀")

In [ ]:
tb = next(b for b in boxes if b["label"] == "table")
x1, y1, x2, y2 = map(int, tb["coordinate"])
page.crop((x1 - 10, y1 - 10, x2 + 10, y2 + 10)).save("table_crop.png")
display(Image.open("table_crop.png").resize(((x2 - x1 + 20) // 2, (y2 - y1 + 20) // 2)))
if vl is not None:
    t0 = time.time()
    for res in vl.predict("table_crop.png"):
        res.save_to_markdown("out_crop")
    crop_md = open(glob.glob("out_crop/*.md")[0], encoding="utf-8").read()
    print(f"표만 변환 {time.time() - t0:.0f}초 · 셀 일치율 {match_rate('t1_domestic', html_rows(crop_md) or md_rows(crop_md)):.0%}")
else:
    print("PaddleOCR-VL 없음 — 건너뜀")

## 5. 심화 🟠 Granite-Docling 258M — 영어 전용 VLM
IBM의 Granite-Docling(258M, Apache 2.0, 비게이트)은 페이지를 **DocTags**(구조 태그)로 바꾸는 초소형 VLM입니다. Docling의 VLM 파이프라인 프리셋으로 부릅니다. 모델 카드 기준 **영어 전용**이고, **T4는 bf16을 지원하지 않아 float32로 돌려야** 합니다(bf16이면 `!!!!`만 출력).
- 0절의 `RUN_GRANITE = True`로 바꾸고 0절부터 다시 실행해야 docling이 설치됩니다.
- Paddle과 Docling을 한 런타임에 섞으면 패키지 충돌이 날 수 있어 기본은 꺼 둡니다.

In [ ]:
if RUN_GRANITE:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import VlmConvertOptions, VlmPipelineOptions
    from docling.datamodel.vlm_engine_options import TransformersVlmEngineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.pipeline.vlm_pipeline import VlmPipeline

    # T4: bf16 미지원 → float32. 8비트 로드(기본 True)도 끕니다.
    opts = VlmConvertOptions.from_preset("granite_docling",
                                         engine_options=TransformersVlmEngineOptions(torch_dtype="float32", load_in_8bit=False))
    conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(
        pipeline_cls=VlmPipeline, pipeline_options=VlmPipelineOptions(vlm_options=opts))})
    for src, label in [("https://arxiv.org/pdf/2408.09869", "영어 논문(Docling 보고서) 1쪽"),
                       (fetch("hanbit_rules.pdf"), "한국어 규정집 2쪽")]:
        t0 = time.time()
        pg = (1, 1) if src.startswith("http") else (2, 2)
        g_doc = conv.convert(src, page_range=pg).document
        print(f"===== {label} · {time.time() - t0:.0f}초 =====")
        print(g_doc.export_to_markdown()[:700], "\n")
else:
    print("RUN_GRANITE = False — 건너뜀 (0절에서 True로 바꾸면 실행)")

💡 **해석** — 강사 PC(CPU, float32)에서 영어 논문 1쪽은 약 48초, 한국어 2쪽은 약 20초 걸렸습니다. 영어 논문은 제목·저자·초록이 마크다운 구조로 깔끔하게 나옵니다. 한국어 쪽도 얼핏 그럴듯하지만 자세히 보면 `숙박비 → 속박비`, `당일 → 담알`, `왕복 → 목별`, `원/박 → 원/백`처럼 **그럴듯한 오자**가 곳곳에 섞입니다. 표 구조는 맞는데 머리글 글자가 틀리는 식이라 눈으로 훑어서는 놓치기 쉽습니다. **모델 카드에 한국어가 없으면 품질 보장도 없습니다.** 모델을 고를 때 모델 카드의 *Languages* 항목을 먼저 확인하세요(이론2 p24).

### 🤔 생각해보기

**레이아웃 + OCR 조립**과 **VLM 한 방 변환**은 각각 언제 유리할까요? 속도·비용·정확도·설명 가능성(어디서 틀렸는지 추적) 측면에서 비교해 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **레이아웃 + OCR 조립**: 모델이 작고 빠르며 CPU로도 돕니다. 박스·신뢰도·좌표가 남아 **어디서 틀렸는지 추적**하기 쉽고, 틀린 단계(검출/인식/행 묶기)만 바꿀 수 있습니다. 대량 문서, 양식이 정해진 문서, 근거 위치(하이라이트)를 보여줘야 하는 서비스에 유리합니다. 대신 인식 모델의 언어 품질이 약하면 그대로 오류가 남습니다.
- **VLM 한 방**: 문맥을 보고 읽어서 받침·표 구조까지 정확하고, 코드가 짧습니다. 복잡한 표·손글씨·수식·혼합 레이아웃에 강합니다. 대신 GPU·시간·비용이 들고, **그럴듯하게 틀리는(환각)** 경우 원인을 추적하기 어렵습니다. 숫자 필드는 샘플 검수가 필요합니다.
- 실무에서는 섞어 씁니다. 레이아웃으로 영역을 나누고, **어려운 영역(표·스캔)만 VLM**에 보내는 식입니다. PaddleOCR-VL 자체가 이 구조(레이아웃 → 영역별 VLM)입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 레이아웃 모델은 영역의 '역할'(제목·본문·표·머리글)과 읽기 순서를 준다. 무괘선 표도 생김새로 찾는다.
- 레이아웃 + OCR은 구조를 살리지만, 글자 품질은 인식 모델이 정한다.
- 한국어를 지원하는 VLM(PaddleOCR-VL v1)은 스캔본 표도 정답 수준으로 읽는다. 대가는 GPU와 시간.
- VLM은 언어 지원부터 확인하라. T4는 bf16 불가 → fp32/fp16.

> 다음 노트북: **08_docling_chunking — Docling으로 마크다운을 만들고 검색용 조각으로 자른다**